# Pronóstico del precio de bolsa de energía en Colombia

## Dos modelos, dos preguntas distintas

Este notebook entrega **dos modelos**, porque el precio de bolsa admite dos
preguntas con respuestas de naturaleza muy distinta:

| | Modelo 1 | Modelo 2 |
|---|---|---|
| **Horizonte** | 1 día | 7 días |
| **Qué responde** | ¿cuánto costará mañana? | ¿con qué probabilidad superará X en una semana? |
| **Salida** | un número (COP/kWh) | una distribución (mediana + intervalo + probabilidades) |
| **Uso** | operación diaria | gestión de riesgo, coberturas |

**Por qué dos y no uno.** A un día el precio es predecible como número: el error
baja de 10,0% a 9,1% frente a repetir el precio de hoy, y se acierta la dirección
en dos de cada tres días. A siete días esa capacidad desaparece —el error del
modelo iguala al de la regla ingenua— **pero el modelo sí conserva información
sobre el riesgo**. Obligarlo a dar un solo número desperdicia justo lo que sabe.
Por eso el segundo modelo entrega probabilidades.

A quince días o más no se encontró capacidad predictiva alguna, y eso se documenta
al final: es un resultado, no una omisión.

## Qué se predice

$$y_t^{(h)} = \log P_{t+h} - \log P_t$$

el cambio del log del precio a $h$ días. El nivel se reconstruye como
$\hat P_{t+h} = P_t\,e^{\hat y}$.

Se modela el **cambio** y no el nivel por una razón que condiciona todo lo demás:
si se modela el nivel, un modelo que se limite a copiar el precio de ayer obtiene
un error bajísimo sin haber aprendido nada. Al modelar el cambio, **el rival
—repetir el precio de hoy— ya incorpora toda la persistencia de la serie**, así
que cualquier mejora sobre él es información genuina.

## Las tres ventanas temporales

Ninguna etapa ve datos de la siguiente. Esta disciplina es lo que hace creíbles
los resultados finales.

| Ventana | Fechas | Qué se decide ahí |
|---|---|---|
| Selección | 2015-2019 | qué variables entran |
| Validación | 2020-2022 | cuántas variables, y la ventana de calibración |
| Prueba A | 2023-2025 | primera medición |
| **Prueba B** | **2026** | **medición final, fuera de muestra puro** |

2026 merece una mención aparte: cuando se tomaron todas las decisiones de
modelado, **esos datos no existían todavía**. Es la prueba más exigente posible,
y además fue un año extremo (el precio pasó de 105 a 1.031 COP/kWh por un evento
El Niño severo).

---
# 1. Preparación

In [ ]:
import importlib.util, subprocess, sys

for paquete in ["xgboost", "statsmodels"]:
    if importlib.util.find_spec(paquete) is None:
        print(f"Instalando {paquete}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)
print("Dependencias listas")

In [ ]:
import io, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import adfuller

from sklearn.linear_model import LassoCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

OBJETIVO = "precio_ponderado"

FIN_SELECCION  = pd.Timestamp("2019-12-31")
INI_VALIDACION = pd.Timestamp("2020-01-01")
FIN_VALIDACION = pd.Timestamp("2022-12-31")
INI_PRUEBA     = pd.Timestamp("2023-01-01")
INI_2026       = pd.Timestamp("2026-01-01")

SEMILLA = 42
SEMILLAS = [42, 7, 123, 2024, 99]   # se promedian para reducir varianza

print("Listo")

### Carga del panel

El archivo `panel_diario_ampliado_2026.csv` es el resultado de un proceso previo
que consolida datos de cuatro fuentes. Se documenta aquí para que quede trazable,
aunque el notebook parte del panel ya armado:

| Fuente | Qué aporta | Script |
|---|---|---|
| **XM / SIMEM** (API `pydataxm`) | precio, hidrología, generación por planta, oferta, demanda, disponibilidad | `00_consolidar_panel.py`, `00b_ampliar_panel.py` |
| **NOAA CPC** | anomalías **semanales** de las regiones Niño 1+2, 3, 3.4 y 4 | `00f_enso_semanal.py` |
| **Open-Meteo / ERA5** | precipitación diaria sobre las 8 cuencas mayores | `00h_lluvia_cuencas.py` |
| **XM (curva de oferta)** | estructura de la curva de mérito diaria | `18_curva_merito.py` |

La celda busca el archivo en disco, luego en GitHub, y si no lo encuentra pide
subirlo. Así corre igual en tu computador que en Colab.

In [ ]:
ARCHIVO = "panel_diario_ampliado_2026.csv"
URL = ("https://raw.githubusercontent.com/andrw4-4/"
       "pronostico-precio-bolsa-colombia/main/panel_diario_ampliado_2026.csv")


def cargar_panel():
    for ruta in [Path(ARCHIVO), Path("data") / ARCHIVO, Path("/content") / ARCHIVO,
                 Path("/content/drive/MyDrive") / ARCHIVO]:
        if ruta.exists():
            print(f"Leído del disco: {ruta}")
            return pd.read_csv(ruta, parse_dates=["fecha"], index_col="fecha")
    try:
        d = pd.read_csv(URL, parse_dates=["fecha"], index_col="fecha")
        print("Descargado de GitHub")
        return d
    except Exception as e:
        print(f"No se pudo descargar ({type(e).__name__}). Súbelo a mano.")
    from google.colab import files
    subidos = files.upload()
    return pd.read_csv(io.BytesIO(subidos[next(iter(subidos))]),
                       parse_dates=["fecha"], index_col="fecha")


df = cargar_panel().sort_index().asfreq("D")
print(f"{df.shape[0]} días x {df.shape[1]} variables  |  "
      f"{df.index.min().date()} a {df.index.max().date()}")

---
# 2. La serie que hay que predecir

In [ ]:
precio = df[OBJETIVO]
log_precio = np.log(precio)
dlog = log_precio.diff()

fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
precio.plot(ax=ax[0], color="k", lw=0.7, title="Precio de bolsa (COP/kWh)")
dlog.plot(ax=ax[1], lw=0.5, color="tab:red", title="Cambio diario del log (Δlog)")
for a in ax:
    for f in [INI_VALIDACION, INI_PRUEBA, INI_2026]:
        a.axvline(f, color="gray", ls="--", lw=0.8)
ax[0].text(pd.Timestamp("2017-01-01"), precio.max()*0.9, "selección", color="gray")
ax[0].text(pd.Timestamp("2021-01-01"), precio.max()*0.9, "validación", color="gray")
ax[0].text(pd.Timestamp("2023-06-01"), precio.max()*0.9, "prueba A", color="gray")
ax[0].text(pd.Timestamp("2026-01-15"), precio.max()*0.9, "prueba B", color="gray")
plt.tight_layout(); plt.show()

for nombre, s in [("precio", precio), ("log precio", log_precio), ("Δlog", dlog)]:
    p = adfuller(s.dropna(), autolag="AIC")[1]
    print(f"{nombre:12s} ADF p={p:.4f}  ->  "
          f"{'estacionaria' if p < 0.05 else 'NO estacionaria'}")

**Sobre la prueba ADF.** Rechaza la raíz unitaria incluso en el nivel, así que
formalmente no obligaría a diferenciar. Se diferencia igual, por dos motivos que
la prueba no captura:

1. El nivel tiene picos de más de 2.500 COP/kWh y una varianza que cambia
   drásticamente entre años (no es estacionario en varianza, cosa que la ADF no
   examina).
2. El motivo de fondo, ya mencionado: al modelar el nivel, copiar el precio de
   ayer parece un modelo excelente. Al modelar el cambio, eso es el punto de
   partida, no el logro.

### 2026: el año de prueba

Vale la pena mirarlo aparte, porque condiciona cómo leer todos los resultados.

In [ ]:
p26 = precio[precio.index >= INI_2026].dropna()
mensual = p26.resample("MS").mean()
print(f"2026: {len(p26)} días, de {p26.min():.0f} a {p26.max():.0f} COP/kWh")
print(f"El mes más caro fue {mensual.max()/mensual.min():.1f} veces el más barato\n")
print(mensual.round(0).to_string())

if "nino12_anom" in df.columns:
    print("\nAnomalía de temperatura del Pacífico frente a Ecuador (Niño 1+2):")
    print(df.loc[df.index >= "2025-11-01", "nino12_anom"].resample("MS").mean().round(2).to_string())

2026 fue un evento El Niño severo: el precio se multiplicó por más de siete entre
febrero y agosto. **Es el peor escenario posible para estrenar un modelo**, y por
eso mismo es la prueba que vale.

---
# 3. Limpieza: qué se descarta y por qué

Hay tres tipos de dato ausente en este panel y **cada uno exige un tratamiento
distinto**. Confundirlos es el error más costoso de esta etapa.

In [ ]:
faltantes = df.isna().mean().sort_values(ascending=False)
print("Variables con faltantes (top 15):")
print((faltantes[faltantes > 0].head(15) * 100).round(1).to_string())

EVENTOS = ["vertimientos", "demanda_no_atendida", "dem_no_atendida_noprog",
           "importaciones", "exportaciones",
           "gen_solar", "frac_solar", "gen_eolica", "frac_eolica", "gen_liquidos"]
EVENTOS = [c for c in EVENTOS if c in df.columns]

print("\nComprobación del supuesto de las variables de evento:")
print("(si no hay ceros, el ausente significa 'no ocurrió', no 'no se sabe')")
for c in EVENTOS[:6]:
    s = df[c]
    print(f"  {c:26s} faltante {s.isna().mean():5.1%}  ceros exactos: {(s == 0).sum()}")

**Tipo 1 — Eventos.** `vertimientos`, `demanda_no_atendida`, `importaciones`,
`gen_solar`… Ninguna tiene un solo cero en once años, y sus mínimos positivos
están lejos de cero. Eso confirma que **XM no publica fila cuando el evento no
ocurrió**: el ausente es un cero real. Se rellenan con 0.

En `gen_solar` el razonamiento es el mismo pero más evidente: en 2015 la
generación solar del país era genuinamente cero, porque no había plantas.

**Tipo 2 — La variable no existía.** `precio_escasez_inf` empieza en 2025 (es una
figura regulatoria nueva), las solares en 2021, `emisiones_co2` en 2018. **No son
faltantes aleatorios: son series que arrancan tarde y luego están completas.**

Esto importa mucho más de lo que parece: las técnicas habituales de imputación
(mediana, KNN, MICE) están pensadas para huecos aleatorios. Aplicarlas aquí sería
**inventar historia**. Se descartan si superan el 15% de ausencia.

**Tipo 3 — Huecos de calendario.** Festivos o fallas de publicación, uno o dos
días en series que se mueven despacio. Relleno hacia adelante, que solo mira al
pasado.

In [ ]:
DESPACHO = ["costo_marginal_prog", "precio_oferta_desp", "max_precio_oferta"]

EX_ANTE = ["precio_escasez", "precio_marg_escasez", "precio_escasez_empalmado",
           "disp_declarada", "enficc", "precio_cargo_conf", "cee", "cere", "mc",
           "margen_reserva", "dummy_creg_escasez",
           "disp_decl_hidraulica", "disp_decl_termica", "disp_decl_despacho_central"]

SIN_2026 = ["demanda_no_atendida", "dem_no_atendida_noprog", "demanda_por_or"]
MAX_FALTANTE = 0.15
INICIO_MAXIMO = pd.Timestamp("2015-03-01")

panel = df.drop(columns=[c for c in SIN_2026 if c in df.columns]).copy()
EVENTOS = [c for c in EVENTOS if c in panel.columns]
EX_ANTE = [c for c in EX_ANTE if c in panel.columns]
panel[EVENTOS] = panel[EVENTOS].fillna(0)

# Variables que arrancan tarde: le cuestan la muestra a TODAS las demás
arranque = panel.apply(lambda s: s.first_valid_index())
tardias = [c for c in panel.columns
           if arranque[c] is not None and arranque[c] > INICIO_MAXIMO
           and panel[c].isna().mean() <= MAX_FALTANTE]

faltan = panel.isna().mean()
candidatas = [c for c in panel.columns
              if c != OBJETIVO and c not in DESPACHO and c not in tardias
              and faltan[c] <= MAX_FALTANTE]

print(f"Descartadas por arrancar tarde: {sorted(tardias)}")
print(f"Descartadas por faltantes o despacho: "
      f"{sorted(set(panel.columns) - set(candidatas) - {OBJETIVO})}")
print(f"\nQuedan {len(candidatas)} variables de {panel.shape[1]}")

base = panel[candidatas].ffill()
base["inv_volumen_util_pct"] = 1 / base["volumen_util_pct"].clip(lower=0.02)
ex_ante = [c for c in base.columns if c in EX_ANTE]
ex_post = [c for c in base.columns if c not in EX_ANTE]
print(f"{len(ex_ante)} ex ante (pueden ir con rezago 0), {len(ex_post)} medidas durante el día")

### Dos exclusiones deliberadas

**El bloque de despacho.** `costo_marginal_prog`, `precio_oferta_desp` y
`max_precio_oferta` se excluyen a propósito. El precio de bolsa **es** el costo
marginal del despacho —la correlación entre ambos es 0,97— así que incluirlos
daría un error espectacular y vacío: el modelo copiaría una versión del precio y
el aporte hidrológico, que es la pregunta de fondo, quedaría invisible.

**Las variables de arranque tardío.** Una sola variable que empieza tarde le cuesta
el inicio de la muestra a todas las demás, porque el `dropna()` conjunto exige que
todas tengan dato el mismo día. `vol_pct_el_quimbo` (el embalse entró en operación
a finales de 2015) costaba **diez meses** de la ventana de selección ella sola.

---
# 4. Ingeniería de características

**La regla que gobierna todo:** para predecir el día $t$ solo puede usarse
información disponible antes de ese día.

| Tipo | Rezago mínimo | Razón |
|---|---|---|
| Medidas durante el día (hidrología, demanda, generación) | 1 día | se conocen al cierre |
| *Ex ante* (precio de escasez, disponibilidad declarada, margen de reserva) | 0 días | se publican antes del despacho |
| Curva de mérito | 1 día | la oferta del día $t$ **determina** el precio del día $t$: usarla sin rezago sería circular |

**El detalle técnico que más fugas causa:** toda media móvil se calcula con
`shift(1)` **antes** del `rolling`, nunca centrada. Una media centrada en $t$ usa
datos de $t+1$ en adelante. Es el error más común y el más difícil de detectar
después, porque no rompe nada: simplemente infla los resultados.

In [ ]:
LAGS = [1, 2, 3, 7, 14, 30]
VENTANAS = [7, 30]


def construir_features():
    piezas = {}

    # --- dinámica del propio precio ---
    for k in LAGS:
        piezas[f"dlog_lag{k}"] = dlog.shift(k)
    for w in VENTANAS:
        piezas[f"dlog_sd{w}"] = dlog.shift(1).rolling(w).std()          # volatilidad
        piezas[f"precio_choque{w}"] = (log_precio.shift(1)
                                       - log_precio.shift(1).rolling(w).mean())

    # --- exógenas ---
    for col in ex_ante:
        for k in [0] + LAGS:
            piezas[f"{col}_lag{k}"] = base[col].shift(k)
    for col in ex_post:
        for k in LAGS:
            piezas[f"{col}_lag{k}"] = base[col].shift(k)

    # choque = valor de ayer menos su media móvil; media móvil = nivel "normal"
    for col in base.columns:
        for w in VENTANAS:
            piezas[f"{col}_choque{w}"] = (base[col].shift(1)
                                          - base[col].shift(1).rolling(w).mean())
            piezas[f"{col}_ma{w}"] = base[col].shift(1).rolling(w).mean()

    X = pd.DataFrame(piezas, index=df.index)
    X["es_lunes"]   = (df.index.dayofweek == 0).astype(int)
    X["es_sabado"]  = (df.index.dayofweek == 5).astype(int)
    X["es_domingo"] = (df.index.dayofweek == 6).astype(int)
    return X


X = construir_features()
print(f"{X.shape[1]} características construidas")

### Limpieza de la matriz de características

Tres filtros, calculados **solo con la ventana de selección**:

1. **Constantes** — no pueden explicar nada.
2. **Redundantes** — si dos variables correlacionan más de 0,95 se conserva la
   primera. Los seis rezagos de una misma variable son casi idénticos entre sí.
3. **Combinaciones lineales exactas** — por construcción
   `x_choque7 = x_lag1 − x_ma7`. Con las tres juntas la matriz es singular y la
   regresión no puede separar sus efectos. Se conserva una variable solo si
   aumenta el rango de la matriz.

In [ ]:
def limpiar(X_sel):
    no_const = X_sel.columns[X_sel.std() > 0]
    Xc = X_sel[no_const]

    corr = Xc.corr().abs()
    quitar = set()
    for i, a in enumerate(Xc.columns):
        if a in quitar:
            continue
        for b in Xc.columns[i + 1:]:
            if corr.loc[a, b] > 0.95:
                quitar.add(b)
    no_red = [c for c in Xc.columns if c not in quitar]

    Z = ((Xc[no_red] - Xc[no_red].mean()) / Xc[no_red].std()).values
    elegidas = []
    for i, _ in enumerate(no_red):
        if np.linalg.matrix_rank(Z[:, elegidas + [i]]) > len(elegidas):
            elegidas.append(i)
    return [no_red[i] for i in elegidas]


COLUMNAS = limpiar(X.loc[:FIN_SELECCION].dropna())
print(f"{X.shape[1]} creadas  ->  {len(COLUMNAS)} tras quitar constantes, "
      f"redundantes y combinaciones exactas")

---
# 5. Utilidades comunes a los dos modelos

In [ ]:
def objetivo_h(h):
    return (log_precio.shift(-h) - log_precio).rename("y")


def particionar(h):
    datos = pd.concat([X[COLUMNAS], objetivo_h(h)], axis=1).dropna()
    return datos


def ranquear(X_sel, y_sel, h):
    tscv = TimeSeriesSplit(n_splits=3, gap=h)
    Xs = StandardScaler().fit_transform(X_sel)
    la = LassoCV(cv=tscv, max_iter=20000, random_state=SEMILLA, n_jobs=-1).fit(Xs, y_sel)
    r1 = pd.Series(np.abs(la.coef_), index=X_sel.columns).rank(ascending=False)
    mo = xgb.XGBRegressor(n_estimators=500, max_depth=3, learning_rate=0.03,
                          subsample=0.8, colsample_bytree=0.8, min_child_weight=10,
                          reg_lambda=3.0, random_state=SEMILLA, n_jobs=-1,
                          tree_method="hist").fit(X_sel, y_sel)
    r2 = pd.Series(mo.feature_importances_, index=X_sel.columns).rank(ascending=False)
    return ((r1 + r2) / 2).sort_values()


def diebold_mariano(e1, e2, h=1):
    d = (e1 ** 2 - e2 ** 2).dropna()
    n = len(d)
    if n < 20:
        return np.nan, np.nan
    gamma = [d.var(ddof=0)] + [np.cov(d[k:], d[:n - k])[0, 1] for k in range(1, h)]
    var_d = (gamma[0] + 2 * sum(gamma[1:])) / n
    if var_d <= 0:
        return np.nan, np.nan
    dm = d.mean() / np.sqrt(var_d)
    dm *= np.sqrt(max(1e-9, (n + 1 - 2 * h + h * (h - 1) / n) / n))
    return float(dm), float(2 * (1 - stats.t.cdf(abs(dm), df=n - 1)))


def metricas(y, pred, precio_hoy, precio_fut):
    nivel = precio_hoy * np.exp(pred)
    direccion = (np.nan if (pred == 0).all()
                 else 100 * np.mean(np.sign(pred) == np.sign(y)))
    return {"RMSE_log": float(np.sqrt(np.mean((y - pred) ** 2))),
            "Direccion_%": direccion,
            "MAPE_%": float(100 * np.mean(np.abs(nivel - precio_fut) / precio_fut))}


def contexto(h, prueba):
    hoy = precio.reindex(prueba.index)
    fut = precio.reindex(prueba.index + pd.Timedelta(days=h))
    fut.index = prueba.index
    return hoy, fut, fut.notna()


print("Utilidades listas")

**Sobre el Diebold-Mariano.** Es la prueba que convierte "mi modelo tiene mejor
error" en un resultado defendible: responde si la diferencia es distinguible del
ruido muestral.

Lleva la corrección de Newey-West con $h$ rezagos, que es **obligatoria** aquí: con
horizonte de 7 días, dos pronósticos consecutivos comparten 6 días de información,
así que los datos de prueba no son observaciones independientes. Sin esa corrección
los p-valores salen artificialmente pequeños y se reportan victorias que no
existen.

---
# 6. Modelo 1 — pronóstico a un día

**Pregunta:** ¿cuánto costará mañana?
**Salida:** un número en COP/kWh.
**Rival:** el modelo ingenuo, "mañana costará lo mismo que hoy".

In [ ]:
H1 = 1
datos1 = particionar(H1)

orden1 = ranquear(datos1.loc[:FIN_SELECCION, COLUMNAS],
                  datos1.loc[:FIN_SELECCION, "y"], H1)

print(f"Ventana de selección: {len(datos1.loc[:FIN_SELECCION])} observaciones")
print("\nLas 15 variables mejor posicionadas para un día:")
for i, v in enumerate(orden1.index[:15], 1):
    print(f"  {i:2d}. {v}")

### Cuántas variables usar

Se recorre una escalera de tamaños y se elige el mejor **en la ventana de
validación**, que no se vuelve a usar para nada.

Este paso parece menor y no lo es. Elegir el número de variables mirando el
periodo de prueba —aunque sea sin querer— equivale a probar diez modelos y
reportar el que mejor salió: alguno acierta por azar.

In [ ]:
ESCALERA = [12, 30, 50, 100, 200, None]
PARAMS1 = dict(n_estimators=600, max_depth=2, learning_rate=0.01, subsample=0.9,
               colsample_bytree=1.0, min_child_weight=10, reg_lambda=10.0)


def entrenar_puntual(entrena, prueba, cols, params):
    preds = [xgb.XGBRegressor(**params, random_state=s, n_jobs=-1, tree_method="hist")
             .fit(entrena[cols], entrena["y"]).predict(prueba[cols]) for s in SEMILLAS]
    return pd.Series(np.mean(preds, axis=0), index=prueba.index)


ent_val1 = datos1.loc[:INI_VALIDACION - pd.Timedelta(days=H1 + 1)]
val1 = datos1.loc[INI_VALIDACION:FIN_VALIDACION]
hoy_v, fut_v, ok_v = contexto(H1, val1)

filas = []
for k in ESCALERA:
    cols = list(orden1.index) if k is None else list(orden1.index[:k])
    pred = entrenar_puntual(ent_val1, val1, cols, PARAMS1)
    m = metricas(val1["y"][ok_v], pred[ok_v], hoy_v[ok_v], fut_v[ok_v])
    i = metricas(val1["y"][ok_v], pd.Series(0.0, index=val1.index)[ok_v],
                 hoy_v[ok_v], fut_v[ok_v])
    filas.append({"k": len(cols), "mejora_%": 100*(m["RMSE_log"]/i["RMSE_log"]-1), **m})
    print(f"  k={len(cols):3d}  RMSE {m['RMSE_log']:.4f} "
          f"({filas[-1]['mejora_%']:+6.2f}%)  MAPE {m['MAPE_%']:5.2f}%  "
          f"dir {m['Direccion_%']:.1f}%")

tv = pd.DataFrame(filas)
K1 = int(tv.loc[tv["mejora_%"].idxmin(), "k"])
cols1 = list(orden1.index[:K1]) if K1 < len(orden1) else list(orden1.index)
print(f"\n==> la validación elige {K1} variables")

### Medición final

In [ ]:
res1 = []
for etapa, ini, fin in [("2023-2025", INI_PRUEBA, INI_2026 - pd.Timedelta(days=1)),
                        ("2026", INI_2026, datos1.index.max())]:
    entrena = datos1.loc[:ini - pd.Timedelta(days=H1 + 1)]
    prueba = datos1.loc[ini:fin]
    hoy, fut, ok = contexto(H1, prueba)

    pred = entrenar_puntual(entrena, prueba, cols1, PARAMS1)
    m = metricas(prueba["y"][ok], pred[ok], hoy[ok], fut[ok])
    ing = metricas(prueba["y"][ok], pd.Series(0.0, index=prueba.index)[ok], hoy[ok], fut[ok])
    dm, p = diebold_mariano((prueba["y"] - pred)[ok], prueba["y"][ok], h=H1)

    res1.append({"etapa": etapa, "n": int(ok.sum()), "k": K1,
                 "MAPE_%": m["MAPE_%"], "MAPE_ing_%": ing["MAPE_%"],
                 "Direccion_%": m["Direccion_%"],
                 "mejora_%": 100*(m["RMSE_log"]/ing["RMSE_log"]-1), "DM": dm, "p": p})
    if etapa == "2026":
        pred1_2026, prueba1_2026 = pred, prueba

    print(f"\n{etapa}  ({int(ok.sum())} días, entrena con {len(entrena)})")
    print(f"  MAPE          {m['MAPE_%']:.2f}%   vs ingenuo {ing['MAPE_%']:.2f}%")
    print(f"  RMSE Δlog     {m['RMSE_log']:.4f}  vs {ing['RMSE_log']:.4f}  "
          f"({100*(m['RMSE_log']/ing['RMSE_log']-1):+.2f}%)")
    print(f"  Dirección     {m['Direccion_%']:.1f}%")
    print(f"  Diebold-Mariano  DM={dm:+.3f}  p={p:.4f}  "
          f"{'-> gana el modelo' if p < 0.05 and dm < 0 else '-> sin diferencia'}")

tabla1 = pd.DataFrame(res1)

In [ ]:
hoy, fut, ok = contexto(H1, prueba1_2026)
nivel = hoy * np.exp(pred1_2026)
objetivo = prueba1_2026.index + pd.Timedelta(days=H1)

fig, ax = plt.subplots(2, 1, figsize=(13, 8))
ax[0].plot(objetivo, fut.values, color="k", lw=1, label="real")
ax[0].plot(objetivo, nivel.values, color="tab:red", lw=1, label="modelo")
ax[0].plot(objetivo, hoy.values, color="tab:blue", lw=0.8, ls="--", label="ingenuo")
f = tabla1[tabla1.etapa == "2026"].iloc[0]
ax[0].set_title(f"Modelo 1 en 2026 — MAPE {f['MAPE_%']:.1f}% vs "
                f"{f['MAPE_ing_%']:.1f}%, dirección {f['Direccion_%']:.1f}%")
ax[0].set_ylabel("COP/kWh"); ax[0].legend(fontsize=8)

err_m = (fut - nivel).abs().rolling(21).mean()
err_i = (fut - hoy).abs().rolling(21).mean()
ax[1].plot(objetivo, err_m.values, color="tab:red", label="error del modelo")
ax[1].plot(objetivo, err_i.values, color="tab:blue", ls="--", label="error del ingenuo")
ax[1].set_title("Error absoluto medio móvil de 21 días")
ax[1].set_ylabel("COP/kWh"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Qué variables usa, y por qué importa

La importancia por permutación se mide **sobre datos de prueba**: cuánto empeora
el error al desordenar una variable. Es más confiable que la ganancia interna de
XGBoost, que se calcula en entrenamiento y premia variables que el modelo usó
mucho aunque no generalicen.

In [ ]:
from sklearn.inspection import permutation_importance

modelo1 = xgb.XGBRegressor(**PARAMS1, random_state=SEMILLA, n_jobs=-1,
                           tree_method="hist")
entrena_final = datos1.loc[:INI_2026 - pd.Timedelta(days=H1 + 1)]
modelo1.fit(entrena_final[cols1], entrena_final["y"])

perm = permutation_importance(modelo1, prueba1_2026[cols1], prueba1_2026["y"],
                              n_repeats=10, random_state=SEMILLA,
                              scoring="neg_root_mean_squared_error")
imp = pd.Series(perm.importances_mean, index=cols1).sort_values()

plt.figure(figsize=(9, max(4, 0.3 * min(20, len(cols1)))))
imp.tail(20).plot.barh(color="tab:blue")
plt.title("Importancia por permutación (medida en 2026)")
plt.tight_layout(); plt.show()

---
# 7. Modelo 2 — pronóstico probabilístico a siete días

## Por qué aquí se cambia la pregunta

Si se le pide a este modelo un solo número para dentro de siete días, empata con
la regla ingenua: **MAPE 28,7% contra 28,7%**. Medido así, no sirve.

Pero el empate en el número esconde algo. El modelo **sí** tiene información,
solo que sobre la *dispersión* y no sobre el *centro*: sabe cuándo hay más riesgo
de que el precio se dispare, aunque no sepa hacia dónde va la mediana. Obligarlo a
dar un punto tira justo eso a la basura.

Así que la pregunta cambia:

> ~~¿Cuánto costará el 15 de abril?~~
> **¿Con qué probabilidad el precio superará 500 COP/kWh el 15 de abril?**

Para decidir un mantenimiento o comprar una cobertura, la segunda es la que se
necesita, y además es contestable.

## Cómo se construye

Se entrena **un modelo por cuantil** con pérdida pinball, que penaliza
asimétricamente: el modelo del percentil 90 paga caro quedarse corto y barato
pasarse, así que aprende el borde superior de la distribución.

## Contra qué se compara

El rival no es el ingenuo puntual sino su versión probabilística: **la
climatología**, es decir "el precio dentro de 7 días será el de hoy multiplicado
por un cambio sacado al azar de la distribución histórica de cambios a 7 días".
Es el análogo probabilístico de la caminata aleatoria, y es un rival duro.

In [ ]:
H7 = 7
CUANTILES = [0.10, 0.25, 0.50, 0.75, 0.90]
NIVEL = 0.80
PARAMS7 = dict(n_estimators=400, max_depth=2, learning_rate=0.01, subsample=0.7,
               colsample_bytree=1.0, min_child_weight=10, reg_lambda=3.0)

datos7 = particionar(H7)
orden7 = ranquear(datos7.loc[:FIN_SELECCION, COLUMNAS],
                  datos7.loc[:FIN_SELECCION, "y"], H7)


def pinball(y, pred, q):
    d = y - pred
    return float(np.mean(np.maximum(q * d, (q - 1) * d)))


def entrenar_cuantiles(entrena, cols, params):
    return {q: xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=q,
                                random_state=SEMILLA, n_jobs=-1,
                                tree_method="hist", **params)
            .fit(entrena[cols], entrena["y"]) for q in CUANTILES}


def predecir_cuantiles(modelos, datos, cols):
    d = pd.DataFrame({q: m.predict(datos[cols]) for q, m in modelos.items()},
                     index=datos.index)
    d.values.sort(axis=1)     # los cuantiles no pueden cruzarse
    return d


def climatologia(entrena, prueba):
    h = entrena["y"].values
    return pd.DataFrame({q: np.full(len(prueba), np.quantile(h, q)) for q in CUANTILES},
                        index=prueba.index)


print("Las 15 variables mejor posicionadas para siete días:")
for i, v in enumerate(orden7.index[:15], 1):
    print(f"  {i:2d}. {v}")

In [ ]:
ent_val7 = datos7.loc[:INI_VALIDACION - pd.Timedelta(days=H7 + 1)]
val7 = datos7.loc[INI_VALIDACION:FIN_VALIDACION]

mejor_k, mejor_pin = None, np.inf
for k in ESCALERA:
    cols = list(orden7.index) if k is None else list(orden7.index[:k])
    qp = predecir_cuantiles(entrenar_cuantiles(ent_val7, cols, PARAMS7), val7, cols)
    pin = np.mean([pinball(val7["y"], qp[q], q) for q in CUANTILES])
    print(f"  k={len(cols):3d}  pérdida pinball {pin:.5f}")
    if pin < mejor_pin:
        mejor_k, mejor_pin = len(cols), pin

K7 = mejor_k
cols7 = list(orden7.index[:K7]) if K7 < len(orden7) else list(orden7.index)
print(f"\n==> la validación elige {K7} variables")

## Calibración: que el 80% signifique 80%

Los cuantiles que produce XGBoost salen **mal calibrados**: el intervalo que dice
80% cubre alrededor del 70%. Un intervalo así es peor que no tener intervalo,
porque da falsa confianza justo cuando se usa para decidir.

La corrección es **conformal cuantílico adaptativo**. La idea es directa: se mide
cuánto se equivocó el intervalo en el pasado reciente y se ensancha lo necesario.
Para cada día se calcula el score de conformidad de los casos ya resueltos:

$$E = \max\big(q_{\text{bajo}}(x) - y,\; y - q_{\text{alto}}(x)\big)$$

$E$ es positivo cuando el valor real quedó fuera del intervalo, y su magnitud dice
por cuánto. El cuantil $(1-\alpha)$ de esos $E$ es lo que hay que sumar y restar.

**Por qué adaptativo y no una corrección fija.** El conformal clásico supone que
el futuro se parece al pasado, y aquí eso no se cumple: 2023-2026 tuvo colas más
gruesas que 2015-2022, hasta el punto de que incluso la climatología se queda
corta. Una corrección calculada de una vez en 2020-2022 volvería a fallar en 2026.
La versión adaptativa recalibra con una ventana móvil y se ajusta sola cuando
cambia el régimen.

**Anti-fuga.** Para predecir $y_t$ solo se conocen los desenlaces de los orígenes
$s$ con $s + h \le t$. La ventana de calibración en $t$ es $[t-h-N,\; t-h]$: nunca
entra un caso cuyo resultado todavía no ocurrió.

In [ ]:
def calibrar(q_hist, y_hist, indice, ventana):
    lo, hi = CUANTILES[0], CUANTILES[-1]
    score = np.maximum(q_hist[lo] - y_hist, y_hist - q_hist[hi])

    aj = pd.Series(index=indice, dtype=float)
    for t in indice:
        fin = t - pd.Timedelta(days=H7)        # último origen ya resuelto en t
        s = score.loc[(score.index > fin - pd.Timedelta(days=ventana))
                      & (score.index <= fin)].dropna()
        if len(s) < 20:
            aj[t] = 0.0
            continue
        n = len(s)
        aj[t] = float(np.quantile(s, min(1.0, np.ceil((n + 1) * NIVEL) / n)))
    return aj


def prob_supera(qp, aj, objetivo):
    qs = np.array(CUANTILES)
    M = qp[CUANTILES].values.copy()
    M[:, 0] -= aj.values
    M[:, -1] += aj.values
    M.sort(axis=1)
    obj = np.asarray(objetivo, dtype=float)
    return pd.Series([1.0 - np.interp(obj[i], M[i], qs, left=0.0, right=1.0)
                      for i in range(len(obj))], index=qp.index)


# la ventana de calibración también se elige en validación
mod_v = entrenar_cuantiles(ent_val7, cols7, PARAMS7)
hist_v = datos7.loc[:FIN_VALIDACION]
q_hist_v = predecir_cuantiles(mod_v, hist_v, cols7)
qp_v = predecir_cuantiles(mod_v, val7, cols7)

mejor_w, mejor_err = None, np.inf
for w in [90, 180, 365]:
    aj = calibrar(q_hist_v, hist_v["y"], val7.index, w)
    cob = float(((val7["y"] >= qp_v[CUANTILES[0]] - aj)
                 & (val7["y"] <= qp_v[CUANTILES[-1]] + aj)).mean())
    print(f"  ventana {w:3d} días  ->  cobertura {cob:.1%}")
    if abs(cob - NIVEL) < mejor_err:
        mejor_w, mejor_err = w, abs(cob - NIVEL)
VENTANA_CAL = mejor_w
print(f"\n==> ventana de calibración elegida: {VENTANA_CAL} días")

### Medición final del modelo 2

Tres métricas, cada una respondiendo algo distinto:

- **Pérdida pinball** — la regla de puntuación propia para cuantiles. Es la
  métrica correcta, no el RMSE.
- **Cobertura** — ¿el intervalo del 80% contiene el 80% de los casos?
- **Brier** — para $P(\text{precio} > \text{umbral})$: mide si las probabilidades
  están *calibradas*, no solo si aciertan.

In [ ]:
UMBRALES = [300, 500, 800]
res7 = []

for etapa, ini, fin in [("2023-2025", INI_PRUEBA, INI_2026 - pd.Timedelta(days=1)),
                        ("2026", INI_2026, datos7.index.max())]:
    entrena = datos7.loc[:ini - pd.Timedelta(days=H7 + 1)]
    prueba = datos7.loc[ini:fin]
    modelos = entrenar_cuantiles(entrena, cols7, PARAMS7)

    hist = datos7.loc[:fin]
    aj = calibrar(predecir_cuantiles(modelos, hist, cols7), hist["y"],
                  prueba.index, VENTANA_CAL)
    qp = predecir_cuantiles(modelos, prueba, cols7)
    qc = climatologia(entrena, prueba)

    hoy, fut, ok = contexto(H7, prueba)
    lo, hi = qp[CUANTILES[0]] - aj, qp[CUANTILES[-1]] + aj

    pin_m = np.mean([pinball(prueba["y"], qp[q], q) for q in CUANTILES])
    pin_c = np.mean([pinball(prueba["y"], qc[q], q) for q in CUANTILES])
    cob_sin = float(((prueba["y"] >= qp[CUANTILES[0]])
                     & (prueba["y"] <= qp[CUANTILES[-1]])).mean())
    cob_cal = float(((prueba["y"] >= lo) & (prueba["y"] <= hi)).mean())

    fila = {"etapa": etapa, "n": len(prueba), "pinball": pin_m, "pinball_clima": pin_c,
            "mejora_pinball_%": 100 * (pin_m / pin_c - 1),
            "cobertura_sin": cob_sin, "cobertura_cal": cob_cal,
            "ancho_cop": float(np.mean(hoy[ok] * (np.exp(hi[ok]) - np.exp(lo[ok]))))}

    print(f"\n{etapa}  ({len(prueba)} días)")
    print(f"  pinball        modelo {pin_m:.5f}   climatología {pin_c:.5f}   "
          f"({fila['mejora_pinball_%']:+.1f}%)")
    print(f"  cobertura 80%  sin calibrar {cob_sin:.1%}  ->  calibrado {cob_cal:.1%}")
    print(f"  ancho medio del intervalo: {fila['ancho_cop']:.0f} COP/kWh")

    for u in UMBRALES:
        obj = np.log(u) - np.log(hoy)
        pm = prob_supera(qp, aj, obj)
        pc = prob_supera(qc, pd.Series(0.0, index=prueba.index), obj)
        real = (fut > u).astype(float)
        bm = float(np.mean((pm[ok] - real[ok]) ** 2))
        bc = float(np.mean((pc[ok] - real[ok]) ** 2))
        fila[f"brier_{u}"] = bm
        print(f"  Brier P(>{u:4d})  modelo {bm:.4f}  climatología {bc:.4f}  "
              f"({100*(bm/bc-1):+.1f}%)")

    res7.append(fila)
    if etapa == "2026":
        qp7_2026, aj7_2026, prueba7_2026 = qp, aj, prueba

tabla7 = pd.DataFrame(res7)

In [ ]:
hoy, fut, ok = contexto(H7, prueba7_2026)
objetivo = prueba7_2026.index + pd.Timedelta(days=H7)
lo = hoy.values * np.exp(qp7_2026[CUANTILES[0]].values - aj7_2026.values)
hi = hoy.values * np.exp(qp7_2026[CUANTILES[-1]].values + aj7_2026.values)
med = hoy.values * np.exp(qp7_2026[0.50].values)

fig, ax = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
ax[0].fill_between(objetivo, lo, hi, alpha=0.3, color="tab:red",
                   label="intervalo 80% calibrado")
ax[0].plot(objetivo, med, color="tab:red", lw=1, label="mediana")
ax[0].plot(objetivo, fut.values, color="k", lw=1.2, label="real")
f = tabla7[tabla7.etapa == "2026"].iloc[0]
ax[0].set_title(f"Modelo 2 en 2026 — cobertura empírica {f['cobertura_cal']:.1%} "
                f"(objetivo 80%)")
ax[0].set_ylabel("COP/kWh"); ax[0].legend(fontsize=8)

obj500 = np.log(500) - np.log(hoy)
p500 = prob_supera(qp7_2026, aj7_2026, obj500)
ax[1].plot(objetivo, p500.values, color="tab:purple", lw=1.2, label="P(precio > 500)")
ax[1].plot(objetivo, (fut.values > 500).astype(float), color="k", lw=0.8,
           alpha=0.4, label="ocurrió")
ax[1].set_ylabel("probabilidad"); ax[1].legend(fontsize=8)
ax[1].set_title("La salida que sirve para decidir")
plt.tight_layout(); plt.show()

### Cómo se usa

In [ ]:
print(f"{'fecha':12s} {'hoy':>7s}   pronóstico a 7 días            P(>500)   real")
print("-" * 76)
for t in prueba7_2026.index[-8:]:
    p0 = precio[t]
    lo_t = p0 * np.exp(qp7_2026.loc[t, CUANTILES[0]] - aj7_2026[t])
    med_t = p0 * np.exp(qp7_2026.loc[t, 0.50])
    hi_t = p0 * np.exp(qp7_2026.loc[t, CUANTILES[-1]] + aj7_2026[t])
    pr = prob_supera(qp7_2026.loc[[t]], aj7_2026.loc[[t]],
                     pd.Series([np.log(500) - np.log(p0)], index=[t]))[t]
    real = precio.get(t + pd.Timedelta(days=H7), np.nan)
    dentro = "dentro" if lo_t <= real <= hi_t else "FUERA"
    print(f"{str(t.date()):12s} {p0:7.0f}   mediana {med_t:6.0f}  "
          f"[{lo_t:5.0f}, {hi_t:6.0f}]   {pr:5.0%}   {real:6.0f} {dentro}")

---
# 8. Resultados de los dos modelos

In [ ]:
print("=" * 78)
print("MODELO 1 — un día, pronóstico puntual")
print("=" * 78)
print(tabla1[["etapa", "n", "k", "MAPE_%", "MAPE_ing_%", "Direccion_%",
              "mejora_%", "p"]].round(3).to_string(index=False))

print("\n" + "=" * 78)
print("MODELO 2 — siete días, pronóstico probabilístico")
print("=" * 78)
print(tabla7[["etapa", "n", "pinball", "pinball_clima", "mejora_pinball_%",
              "cobertura_sin", "cobertura_cal", "ancho_cop"]].round(4).to_string(index=False))

f1 = tabla1[tabla1.etapa == "2026"].iloc[0]
f7 = tabla7[tabla7.etapa == "2026"].iloc[0]
print("\n" + "=" * 78)
print("VEREDICTO SOBRE 2026 (datos que no existían al construir los modelos)")
print("=" * 78)
print(f"  Modelo 1:  MAPE {f1['MAPE_%']:.2f}% vs {f1['MAPE_ing_%']:.2f}% del ingenuo, "
      f"dirección {f1['Direccion_%']:.1f}%, p={f1['p']:.4f}")
print(f"  Modelo 2:  pinball {f7['mejora_pinball_%']:+.1f}% vs climatología, "
      f"cobertura {f7['cobertura_cal']:.1%} (objetivo 80%)")

---
# 9. Lo que no funcionó

Esta sección importa tanto como las anteriores. Se probaron sistemáticamente seis
vías para extender el pronóstico más allá de una semana, **y ninguna funcionó**.
Documentarlo evita que alguien repita el camino, y es en sí mismo un resultado.

| Intervención | Resultado en 2026 |
|---|---|
| Ampliar el panel de 58 a 104 variables | sin efecto (RMSE 0,1641 → 0,1642) |
| Bloque ENSO semanal | corrigió el fallo grave a 30-90 días (dirección de 27% a 38%), pero sin llegar a ganar |
| Precipitación sobre las cuencas (ERA5) | sin efecto |
| Reducir a pocas variables | **peor**: el óptimo está en 30-100, no en 5 |
| Extender la historia a 2000-2014 (26 años, ~10 eventos ENSO) | **no ayuda**; a 30 días empeora |
| Pronóstico puntual a 15, 30, 60 y 90 días | peor que la regla ingenua en todos los casos |

### Los tres hallazgos metodológicos

**1. El horizonte útil es corto.** A 30 días el modelo tiene 52% de error contra
41% de simplemente repetir el precio de hoy. A 90 días acierta la dirección el
38,5% de las veces, peor que una moneda.

**2. Más información no es mejor modelo.** Tres bloques distintos de datos nuevos
—mezcla de generación, clima semanal, lluvia por cuenca— no movieron el resultado.
La señal es débil y está repartida: ninguna variable manda, y agregar más la
diluye.

**3. Los resultados favorables no sobreviven a la validación honesta.** Ocurrió
tres veces en este trabajo:

- Los horizontes de 60 y 90 días se veían bien en 2023-2025 (−11% y −14% frente
  al ingenuo). En 2026 la dirección cayó a 34% y 27%: el modelo quedó
  sistemáticamente invertido.
- Un p-valor de 0,0496 a siete días dejó de ser significativo (0,08) al promediar
  varias semillas.
- Elegir el número de variables mirando el periodo de prueba producía una mejora
  que desaparecía al elegirlo en validación.

Esa última lección es la que más vale: en la literatura de pronóstico de precios
de energía abundan las mejoras reportadas que nunca se revalidaron en un periodo
nuevo. Este trabajo muestra, con datos, por qué eso es un problema.

### Qué exploraría a continuación

- **Modelo estructural de despacho.** El precio *es* el costo marginal del
  despacho (correlación 0,97). Reconstruir la curva de mérito y simular el
  despacho no requiere haber *visto* una sequía para valorarla: la calcula. Es la
  debilidad de fondo del enfoque estadístico y la razón por la que la industria
  usa modelos hidrotérmicos.
- **Extender el enfoque probabilístico a 30 días.** A ese horizonte solo funcionó
  el umbral extremo, $P(\text{precio} > 800)$, donde el modelo le gana a la
  climatología ~5% en ambos periodos. Tiene sentido físico: los precios extremos
  los causan fundamentales observables, mientras que las fluctuaciones normales
  son ruido.